# RAGWire Setup and First Retrieval

**Stack:** Ollama · Qdrant (local)

Progressive journey:
- Ingest a single document and retrieve chunks
- Scale to multiple companies
- Explore metadata and add filters
- Build an agent — simple first, then filter-aware

## 1. Setup

In [24]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

from ragwire import RAGWire, setup_logging
import ragwire

print(ragwire.__version__)


1.6.1


In [25]:
logger = setup_logging(log_level="INFO")

## 2. Ingest a Single Document and Retrieve

**Model:** `qwen3.5:9b` · **Embedding:** `qwen3-embedding:0.6b` · **Vector Store:** Qdrant (local)

In [26]:
rag = RAGWire(config_path="config.yaml")

2026-09-21 09:34:16,742 - ragwire.core.pipeline - INFO - Loading configuration from config.yaml
2026-09-21 09:34:16,882 - ragwire.core.pipeline - INFO - Document loader initialized
2026-09-21 09:34:16,884 - ragwire.core.pipeline - INFO - Text splitter initialized (strategy=markdown, chunk_size=10000)
2026-09-21 09:34:16,885 - ragwire.core.pipeline - INFO - Ingestion configured (workers=1, batch_size=64, retries=2, replace_changed=True)
2026-09-21 09:34:18,119 - ragwire.core.pipeline - INFO - Embedding model initialized (provider=ollama)
2026-09-21 09:34:19,231 - ragwire.core.pipeline - INFO - LLM initialized for metadata extraction (provider=ollama, model=qwen3.5:9b)
2026-09-21 09:34:19,785 - ragwire.vectorstores.qdrant_store - INFO - Connected to Qdrant at http://localhost:6333
2026-09-21 09:34:19,819 - ragwire.vectorstores.qdrant_store - INFO - Deleted collection: finance-rag-ollama
2026-09-21 09:34:19,821 - ragwire.core.pipeline - INFO - Deleted existing collection for recreation: f

In [28]:
stats = rag.ingest_documents(["../data/finance_data/Apple_10k_2025.pdf"])

2026-09-21 09:42:26,595 - ragwire.core.pipeline - INFO - Starting ingestion of 1 documents


Ingesting:   0%|          | 0/1 [15:47<?, ?file/s]


KeyboardInterrupt: 

# RAGWire and Agentic RAG
- Upload set of documents
- Retrieval 
- Agentic RAG

## 3. Scale to Multiple Companies

Ingest all three 10-K filings at once. RAGWire deduplicates — re-running skips already-ingested files.

## 4. Explore Metadata

RAGWire extracts company name, doc type, and fiscal year during ingestion. Let's inspect what's stored.

## 5. Manual Metadata Filters

The simple agent above can mix up companies when all three are in the same collection.
Filters let us pin retrieval to a specific company, year, or doc type.

## 6. Auto-Filter

RAGWire can extract filters from the query automatically — no need to pass them manually.

## 7. Simple Agent — No Filters

Start with a basic agent that has one tool: `search_documents`. No metadata awareness yet.

## 8. Filter-Aware Agent

Upgrade the agent with two tools: `get_filter_context` (inspect what's stored) and `search_documents` (retrieve with optional filters).
The model decides when to use filters based on the query.

We also upgrade the model to `qwen3.5:27b` for better multi-document reasoning.

## 9. Interactive Q&A Loop

Put it all together — the filter-aware agent in a conversational loop.